In [0]:
from pyspark.sql import functions as f

In [0]:
INGESTION_CONFIG = [
    {
        "source": "S3 bucket",
        "path": "s3://sportpro-db/customers/*.csv",
        "table": "customers"
    },
    {
        "source": "S3 bucket",
        "path": "s3://sportpro-db/products/*.csv",
        "table": "products"
    },
    {
        "source": "S3 bucket",
        "path": "s3://sportpro-db/gross_price/*.csv",
        "table": "gross_price"
    },
]

#Ingest Files into Bronze Tables

In [0]:
for item in INGESTION_CONFIG:
    print(f"Ingesting {item['source']} ---> pcat.bronze.{item['table']}")
    
    df = (
        spark.read 
            .option("header", "true")
            .option("inferSchema", "true")
            .csv(item['path'])
            .withColumn("read_timestamp", f.current_timestamp())
            .select("*", "_metadata.file_name", "_metadata.file_size")
    )
    
    df.write.mode("overwrite") \
        .option("delta.enableChangeDataFeed", "true") \
        .saveAsTable(f"pcat.bronze.{item['table']}")